In [9]:
"""
Validação completa do Data Lake S3 — forest-risk-datalake
Corre numa célula do Jupyter.
"""
import boto3
import pandas as pd
import io

s3 = boto3.client(
    's3',
    endpoint_url='http://localstack:4566',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

BUCKET = 'forest-risk-datalake'

def listar_parquets(prefixo):
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefixo, MaxKeys=1000)
    return [o for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')]

def tamanho_total(ficheiros):
    return sum(o['Size'] for o in ficheiros) / 1024  # KB

def ler_amostra(ficheiros):
    """Le o primeiro Parquet disponivel para inspecionar colunas e conteudo."""
    if not ficheiros:
        return None
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=ficheiros[0]['Key'])
        return pd.read_parquet(io.BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"   Erro a ler amostra: {e}")
        return None

SEP = "=" * 65

print(SEP)
print("VALIDAÇÃO DO DATA LAKE — forest-risk-datalake")
print(SEP)

resultados = {}

# ══════════════════════════════════════════════════════════════════
# 1. HISTÓRICO NASA FIRMS (hotspots/)
# ══════════════════════════════════════════════════════════════════
historico = listar_parquets('hotspots/')
print(f"\n📂 HISTÓRICO NASA FIRMS (hotspots/)")
print(f"   Ficheiros Parquet : {len(historico)}")
print(f"   Tamanho total     : {tamanho_total(historico):.1f} KB")

anos, zonas = set(), set()
for o in historico:
    for p in o['Key'].split('/'):
        if p.startswith('ano='):     anos.add(p.replace('ano=', ''))
        if p.startswith('grid_id='): zonas.add(p.replace('grid_id=', ''))

print(f"   Anos cobertos     : {sorted(anos)}")
print(f"   Zonas cobertas    : {len(zonas)} zonas")

if historico:
    df_h = ler_amostra(historico)
    if df_h is not None:
        print(f"   Colunas           : {list(df_h.columns)}")
        # Verifica colunas essenciais
        cols_esperadas = ['latitude', 'longitude', 'frp', 'acq_date', 'confidence']
        faltam = [c for c in cols_esperadas if c not in df_h.columns]
        if faltam:
            print(f"   ⚠️  Colunas em falta : {faltam}")
        else:
            print(f"   Colunas essenciais: ✅ todas presentes")

resultados['historico_nasa'] = len(historico) > 0 and len(anos) >= 3

# ══════════════════════════════════════════════════════════════════
# 2. METEOROLOGIA ERA5 (meteorologia/)
# ══════════════════════════════════════════════════════════════════
meteo = listar_parquets('meteorologia/')
print(f"\n🌤️  METEOROLOGIA ERA5 (meteorologia/)")
print(f"   Ficheiros Parquet : {len(meteo)}")
print(f"   Tamanho total     : {tamanho_total(meteo):.1f} KB")

if meteo:
    anos_m = set()
    for o in meteo:
        for p in o['Key'].split('/'):
            if p.startswith('ano='): anos_m.add(p.replace('ano=', ''))
    print(f"   Anos cobertos     : {sorted(anos_m)}")
    df_m = ler_amostra(meteo)
    if df_m is not None:
        print(f"   Colunas           : {list(df_m.columns)}")
        cols_era5 = ['temp_c', 'rh', 'wind_speed_kmh']
        faltam = [c for c in cols_era5 if c not in df_m.columns]
        if faltam:
            print(f"   ⚠️  Colunas em falta : {faltam}")
        else:
            print(f"   Colunas essenciais: ✅ todas presentes")
else:
    print(f"   ⏳ Vazio — EDA_ERA5.py ainda nao correu")

resultados['meteo_era5'] = len(meteo) > 0

# ══════════════════════════════════════════════════════════════════
# 3. STREAMING SPARK — sensor-events (agregados_streaming/)
# ══════════════════════════════════════════════════════════════════
streaming = listar_parquets('agregados_streaming/')
streaming = [o for o in streaming if '_spark_metadata' not in o['Key']]
print(f"\n📡 STREAMING SPARK (agregados_streaming/)")
print(f"   Ficheiros Parquet : {len(streaming)}")
print(f"   Tamanho total     : {tamanho_total(streaming):.1f} KB")

if streaming:
    df_s = ler_amostra(streaming)
    if df_s is not None:
        print(f"   Colunas           : {list(df_s.columns)}")
        print(f"   Linhas na amostra : {len(df_s)}")
        # Verifica se tem o join dos 3 streams (versao nova)
        cols_join = ['n_hotspots', 'frp_medio', 'risco_composto']
        tem_join = all(c in df_s.columns for c in cols_join)
        if tem_join:
            print(f"   Join 3 streams    : ✅ risco_composto presente")
        else:
            print(f"   Join 3 streams    : ⚠️  colunas do join em falta {[c for c in cols_join if c not in df_s.columns]}")
            print(f"                       (job antigo ainda a correr — reinicia spark-streaming)")
        if 'grid_id' in df_s.columns:
            print(f"   Zonas na amostra  : {df_s['grid_id'].unique().tolist()}")
        if 'risco_composto' in df_s.columns:
            print(f"   Risco composto    : min={df_s['risco_composto'].min():.1f} max={df_s['risco_composto'].max():.1f}")
else:
    print(f"   ⏳ Vazio — a aguardar janelas fecharem (modo append ~10-15 min)")

resultados['streaming'] = len(streaming) > 0

# ══════════════════════════════════════════════════════════════════
# 4. CHECKPOINTS (saude do Spark streaming)
# ══════════════════════════════════════════════════════════════════
checkpoints = s3.list_objects_v2(Bucket=BUCKET, Prefix='checkpoints/')
n_ckpt = checkpoints.get('KeyCount', 0)
print(f"\n🔧 CHECKPOINTS SPARK")
print(f"   Ficheiros         : {n_ckpt}")
if n_ckpt > 0:
    print(f"   Estado            : ✅ Spark a guardar progresso")
else:
    print(f"   Estado            : ⏳ Streaming ainda nao iniciou")

resultados['checkpoints'] = n_ckpt > 0

# ══════════════════════════════════════════════════════════════════
# 5. TODOS OS BUCKETS / PREFIXOS
# ══════════════════════════════════════════════════════════════════
print(f"\n📋 PREFIXOS NO BUCKET")
todos = s3.list_objects_v2(Bucket=BUCKET, Delimiter='/')
for p in todos.get('CommonPrefixes', []):
    objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=p['Prefix'])
    n = objs.get('KeyCount', 0)
    print(f"   {p['Prefix']:<35} {n} objectos")

# ══════════════════════════════════════════════════════════════════
# RESUMO FINAL
# ══════════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("RESUMO")
print(SEP)
print(f"  Histórico NASA (hotspots)    : {'✅ OK' if resultados['historico_nasa'] else '❌ VAZIO — corre carga_historico_s3.py'}")
print(f"  Meteorologia ERA5            : {'✅ OK' if resultados['meteo_era5'] else '⏳ VAZIO — EDA_ERA5.py nao correu'}")
print(f"  Streaming Spark              : {'✅ OK' if resultados['streaming'] else '⏳ A aguardar — janelas ainda a fechar'}")
print(f"  Checkpoints Spark            : {'✅ OK' if resultados['checkpoints'] else '⏳ Streaming nao iniciou'}")

n_ok = sum(resultados.values())
n_total = len(resultados)

print(f"\n  {n_ok}/{n_total} componentes OK", end="  ")
if n_ok == n_total:
    print("✅ DATA LAKE COMPLETO")
elif n_ok >= 2:
    print("🟡 DATA LAKE PARCIAL")
else:
    print("❌ DATA LAKE INCOMPLETO")
print(SEP)

VALIDAÇÃO DO DATA LAKE — forest-risk-datalake

📂 HISTÓRICO NASA FIRMS (hotspots/)
   Ficheiros Parquet : 580
   Tamanho total     : 7834.8 KB
   Anos cobertos     : ['2020', '2021', '2022', '2023', '2024']
   Zonas cobertas    : 10 zonas
   Colunas           : ['latitude', 'longitude', 'bright_ti4', 'bright_ti5', 'frp', 'acq_date', 'acq_time', 'confidence', 'daynight', 'satellite', 'satelite', 'dia']
   Colunas essenciais: ✅ todas presentes

🌤️  METEOROLOGIA ERA5 (meteorologia/)
   Ficheiros Parquet : 0
   Tamanho total     : 0.0 KB
   ⏳ Vazio — EDA_ERA5.py ainda nao correu

📡 STREAMING SPARK (agregados_streaming/)
   Ficheiros Parquet : 10
   Tamanho total     : 17.9 KB
   Colunas           : ['janela_inicio', 'janela_fim', 'grid_id', 'n_leituras_sensor', 'risk_medio_sensor', 'risk_maximo_sensor', 'temp_media', 'humidade_media', 'vento_medio', 'n_hotspots', 'frp_medio', 'frp_maximo', 'temp_max_media', 'humidade_ipma', 'vento_max_ipma', 'precipitacao_media', 'risco_composto']
   Linhas